# Hybrid + local cross-encoder reranker

Iteration 20 builds on the user-selected hybrid with its 0.55 cosine cutoff unchanged. The pretrained cross-encoder scores question/passage pairs and reorders the top 20 fused candidates. This notebook reads saved train results; no model inference or test evaluation is triggered.

In [2]:
from pathlib import Path
import json,sys
import pandas as pd
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'src/retrieval.py').exists())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'scripts'))
from project_paths import source_path
rows=[json.loads(line) for line in (ROOT/'experiments/semantic_log.jsonl').read_text(encoding='utf-8').splitlines()]
run=next(row for row in reversed(rows) if row['iteration']==20)
run['config']

{'name': 'hybrid_minilm_reranked_055',
 'retriever_type': 'reranked_hybrid',
 'base_config': 'configs/hybrid_bm25_minilm_rrf_abstention.json',
 'model_id': 'cross-encoder/ms-marco-MiniLM-L6-v2',
 'revision': '233902d25c440f23af6f7d6e94d2946bac0bee0a',
 'candidate_pool': 20,
 'max_length': 512,
 'batch_size': 16,
 'torch_threads': 4,
 'device': 'cpu',
 'score_type': 'raw_logit',
 'protocol': 1}

## What changed

Only passage ranking changed. The hybrid retrieves up to 20 candidates after applying the existing abstention gate. The local cross-encoder reads each full question with a passage and produces a raw logit, not a probability. Return the top k original passages; source offsets are preserved. There is no new threshold, answer generation or fine-tuning. Model card: https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2 .

In [3]:
table=[]
for name,result in run['metrics'].items():
    m=next(m for m in result['retrieval'] if m['k']==5)
    table.append(dict(version=name,precision=m['precision'],recall=m['recall'],benchmark_precision=m['benchmark_precision'],benchmark_recall=m['benchmark_recall'],hit_rate=m['hit'],correct_abstention=result['correct_abstention'],false_abstention=result['false_abstention']))
pd.DataFrame(table).set_index('version').style.format('{:.2%}')

,precision,recall,benchmark_precision,benchmark_recall,hit_rate,correct_abstention,false_abstention
version,,,,,,,
reference,7.12%,65.51%,7.49%,74.24%,75.00%,39.13%,0.71%
candidate,3.59%,34.89%,3.73%,38.79%,46.43%,39.13%,0.71%


In [4]:
print('Quality checks:',run['checks'])
print('Quality accepted:',run['quality_accepted'])
print('Operational accepted:',run['operational_accepted'])
print('First-use timing:',run['estimated_first_use_latency_ms'])
print('Timing excludes:',run['timing_excludes'])
print('Truncated pairs:',run['truncated_pairs'],'/',run['pair_count'])
print('Test evaluations:',run['test_evaluation_runs'])

Quality checks: {'recall_gain': False, 'precision': False, 'false_abstention': True, 'correct_abstention': True}
Quality accepted: False
Operational accepted: False
First-use timing: {'median': 580.9352499782108, 'p95': 1078.6507849523325}
Timing excludes: One-time model/index load and initial public model download
Truncated pairs: 7 / 3060
Test evaluations: 0


## Inspect a recorded prediction

The reranker scores and the original hybrid scores are kept separately. Reading these saved results requires no model weights.

In [5]:
saved=json.loads((ROOT/run['result_directory']/'candidate.json').read_text(encoding='utf-8'))
example=next(p for p in saved['answerable']['predictions'] if p['hits'])
print(example['query'])
pd.DataFrame(example['hits'])[['file_path','start','end','score','hybrid_score','text']].head(3)

Consider the Amendment No. 3 to Wireless Content License Agreement between Fox Mobile Entertainment, Inc. and Glu Mobile Inc.; What is the expiration date of this contract?


,file_path,start,end,score,hybrid_score,text
0,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,0,1000,8.232897,0.016393,"AMENDMENT NO. 3\n\nDated as of February 19, 20..."
1,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,800,1800,1.416048,0.003968,"dom of Heaven, Mr. and Mrs. Smith, In Her Shoe..."
2,cuad/GluMobileInc_20070319_S-1A_EX-10.09_43663...,1600,1902,-1.224478,0.012097,"x Mobile Entertainment, Inc. (""Fox"")\n\n\n\nBy..."


## Reproduce

Default `python manage.py evaluate_reranker` replays cached logits without model calls. A fresh machine can first run `python manage.py prepare_reranker --download`, then `python manage.py evaluate_reranker --fill` for local inference. Weights are ignored by Git; exact pinned model settings and score caches are retained. Edit config only for a new logged experiment. Presentation interfaces use cache-only replay. Final hybrid + reranker + LLM is the next separate experiment.